# Letting Agents Manage Their Own Context

In NOOA, **agents can manage their own context**. Alongside the tools you write,
you can give an agent access to `self.events`: an API for inspecting its history
and replacing completed work with a concise summary. The model decides when to
do this, which events to collapse, and what information to preserve.

The key operation is **`self.events.collapse()`**. It archives a range of events
and puts a summary in their place in the active history. The original events
remain retrievable by tag, so the agent can return to the details if it needs to.

In this notebook, we simulate a piece of context becoming irrelevant and watch the model compact its own history. To do so, we use a `MovieNightAgent` that helps us choose a movie. We start with space movies, so long plot descriptions pile up in its history. Then we change our request: something funny, with time travel, and under two hours. The earlier plot descriptions are no longer useful, but our preferences and our reasons for rejecting each movie still matter. We never tell the agent which events to collapse; it picks the range and writes the summary itself.

We'll follow the conversation through three steps:

- **Research the first shortlist** — read movie cards through an ordinary Python
  tool and see how the results become part of the agent's history.
- **Change the plan** — let the model collapse the completed research while
  preserving the facts and decisions it still needs.
- **Ask a callback question** — “Why did we rule out the first one?” Check that
  the agent can answer without another catalog lookup.

The movie catalog stays in a private Python field; only requested cards enter
the conversation. We'll inspect the history before and after we change our
request, then ask a follow-up question.

## Prerequisites

Run the install cell below before the setup cell.

The setup cell below lists several providers. Uncomment the one you want to use. For hosted providers, set your key in Colab's Secrets (🔑 in the left sidebar) as `API_KEY`. NOOA is also compatible with **local inference** (Ollama, vLLM, any OpenAI-compatible endpoint) - those need no API key, just an `api_base`.


In [ ]:
!pip install nooa


## Setup

NOOA works with any LiteLLM-supported model - hosted or local. Pick one below. For hosted providers, set your key in Colab's Secrets (🔑 in the sidebar) as `API_KEY`. Local providers such as Ollama and vLLM do not need a key, just an `api_base`.


In [ ]:
from nooa.unifiedllm.registry import get_llm_client

# In Colab: click the 🔑 icon in the left sidebar, add a secret called API_KEY,
# and enable "Notebook access". Then:
from google.colab import userdata
api_key = userdata.get("API_KEY")

# Locally: `export API_KEY=...` in your shell, then uncomment:
# import os
# api_key = os.environ["API_KEY"]

# model = get_llm_client("claude-haiku-4-5", api_key=api_key)                          # Anthropic
model = get_llm_client("gpt-5.5", api_key=api_key)                                     # OpenAI
# model = get_llm_client("ollama_chat/qwen3:1.7b", api_base="http://localhost:11434")   # Ollama (local, no key)
# model = get_llm_client("hosted_vllm/Qwen/Qwen3-1.7B", api_base="http://localhost:8000/v1") # vLLM (local, no key)


## The Movie Catalog

Our agent will read movie cards from a small JSON catalog hosted in the tutorials repository. The cell below downloads it into `assets/` (skipped if the file is already there) and imports what we need. The agent keeps the catalog in a private field, so the model only sees the cards it asks for.

In [ ]:
from nooa import Agent, hidden
from nooa.agentdoc import spec

import json
from html import escape
from pathlib import Path
from urllib.request import urlretrieve
from IPython.display import HTML, display

# Hidden, so the model can't see where the catalog lives and read it directly.
with hidden:
    CATALOG_URL = (
        "https://raw.githubusercontent.com/NVIDIA-NeMo/labs-OO-Agents/main/notebook_tutorials/assets/movie_night_catalog.json"
    )
    CATALOG_PATH = Path("assets/movie_night_catalog.json")

if not CATALOG_PATH.exists():
    CATALOG_PATH.parent.mkdir(exist_ok=True)
    urlretrieve(CATALOG_URL, CATALOG_PATH)

## The Movie-Night Assistant

Our agent reads one movie card at a time through `lookup_movie()`, an ordinary
Python tool. The rest of the catalog stays in `self._catalog`, a private field
hidden from the model.

`suggest()` handles each turn of the conversation. Its docstring asks the
model to condense completed research when the shortlist changes.
**`spec(self, "events", hidden=False)`** exposes the Events API so the model
can inspect its history and call `collapse()` itself.

> 📝 `context` and `events` are hidden from the model by default. The single line `spec(self, "events", hidden=False)` is what opts the Events API in.

In [2]:
class MovieNightAgent(Agent, llm=model):
    """Choose movies using catalog evidence and the group's evolving preferences."""

    def __init__(self):
        super().__init__()
        spec(self, "events", hidden=False)
        self._catalog = json.loads(CATALOG_PATH.read_text())
        self._lookups = []

    def lookup_movie(self, title: str) -> str:
        """Read one catalog card by exact title."""
        if title not in self._catalog:
            raise ValueError(f"Unknown movie title: {title!r}")
        self._lookups.append(title)
        return json.dumps({"title": title, **self._catalog[title]}, indent=2)

    @hidden
    async def suggest(self, request: str) -> str:
        """Help choose a movie. Call lookup_movie() for new candidates and use
        the catalog's runtimes and notes, rather than guessing from memory.
        Give a short, conversational answer. Tone notes are editorial judgments.

        When the shortlist changes, use self.events.collapse() to condense
        completed research before reading new cards. Preserve preferences,
        candidate titles, runtimes, and reasons for keeping or rejecting them.
        Keep the current request outside the collapsed range. Summarize the
        decisions, not the full plots. Inspect history using tags and types;
        printing old event bodies would add copies back into active history.

        For follow-up questions, use retained history. Look up a card again
        only if a needed fact is missing. Respect all current preferences.
        """
        ...

assistant = MovieNightAgent()

The helper below displays the conversation history as a table, with summaries
and expandable lists of archived events. Run the cell once; you can skip
reading its implementation and move on to the conversation.


In [3]:
@hidden
def show_history(events):
    """Display active events, with summaries and expandable archive tags."""
    table = ("width:100%;border-collapse:collapse;background:#e8ecf1;border:1px solid #d0d7de;"
             "color:#24292f;font:13px/1.45 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;")
    cell = "padding:6px 10px;border-bottom:1px solid #d0d7de;text-align:left;vertical-align:top;"
    rows = []
    for tag in events.keys():
        event = events[tag]
        kind = type(event).__name__
        preview = getattr(event, "prompt", "") or getattr(event, "stdout", "")
        detail = escape(" ".join(preview.split())[:100])
        if kind == "Summary":
            archived = "<br>".join(
                f"{escape(child.tag)} · {type(child).__name__}"
                for child in events[event.children_tags]
            )
            detail = (f'<p style="white-space:pre-wrap;margin:0">{escape(event.summary_text or "")}</p>'
                      f"<details><summary>Archived events</summary>{archived}</details>")
        rows.append(f'<tr><td style="{cell}"><code>{escape(tag)}</code></td>'
                    f'<td style="{cell}">{kind}</td><td style="{cell}">{detail}</td></tr>')
    header = "".join(f'<th style="{cell}">{name}</th>' for name in ("Tag", "Event", "Content"))
    display(HTML(f'<table style="{table}"><tr>{header}</tr>' + "".join(rows) + "</table>"))

## Let's Choose a Movie

Let's start with one firm rule: no horror. We'll ask the agent to compare
three space movies. This research is still useful, so it can stay in the
active history for now.


In [4]:
first_answer = await assistant.suggest(
    "Movie night! Our one firm rule is no horror. Let's try something in space. "
    "Compare Interstellar, Gravity, and Alien, in that order. "
    "Tell us the runtimes and which would suit us best."
)
print(first_answer)


- **Interstellar — 169 minutes:** Serious, emotionally intense space drama. No horror, but it’s a substantial evening’s viewing.
- **Gravity — 91 minutes:** A tense space-survival thriller, not horror. Shorter, though the catalog warns of sustained threat and disturbing images.
- **Alien — 117 minutes:** Space horror—ruled out by your firm preference.

**I’d pick Interstellar** for an expansive space movie without horror, provided you’re happy with the long runtime. Choose **Gravity** if you’d rather keep it short and don’t mind sustained tension.

Let's look at the history so far. Each row has an event tag and type.
The movie cards appear in Python execution outputs, so several cards may
share one event.


In [5]:
show_history(assistant.events)

Tag,Event,Content
1,Task,## Task: suggest Help choose a movie. Call lookup_movie() for new candidates and use the catalog's r
2,ToolCallEvent,
3,PythonOutput,"Task: suggest() request (str): ""Movie night! Our one firm rule is no horror. Let's try something in"
4,LLMResponse,
5,ToolCallEvent,
6,PythonOutput,"class EventsApi: """""" Query past events by type, tag, text, or call ID. Compact context by summarizin"
7,LLMResponse,
8,ToolCallEvent,
9,PythonOutput,"{ ""title"": ""Interstellar"", ""year"": 2014, ""runtime_minutes"": 169, ""genres"": [ ""science fiction"", ""spa"
10,LLMResponse,


**“Actually… time travel. And make it funny.”**

Now let's change our request. The earlier plot descriptions can go, but the
agent should remember our preferences and why the first choices don't fit
anymore. Let's see how it makes room for the new shortlist.


In [6]:
second_answer = await assistant.suggest(
    "Actually, let's switch to a light time-travel comedy, strictly under two hours. "
    "Our no-horror rule still stands. Compare Back to the Future, About Time, "
    "and Edge of Tomorrow. Pick one, and remember why the earlier options no longer fit."
)
print(second_answer)

My pick: **Back to the Future (116 minutes)**. It’s a playful time-travel comedy-adventure, fits strictly under two hours, and isn’t horror. The catalog rates it the best mood match—though it does have peril and some dated material.

- **About Time (124 minutes):** warm and romantic, but over your firm limit; also sometimes bittersweet.
- **Edge of Tomorrow (113 minutes):** short enough, but combat-heavy and intense. Comic moments don’t make it a light comedy.

The earlier picks no longer fit: **Interstellar (169 minutes)** is too long and serious; **Gravity (91 minutes)** is tense survival rather than time-travel comedy; **Alien (117 minutes)** still breaks your no-horror rule.

Let's look at the history again. A `Summary` event should now replace the
completed space-movie research, keeping the preferences and decisions that
still matter. Expand **Archived events** to see the tags and types of the
original records.


In [7]:
assert assistant.events.query(type="Summary"), "The agent hasn't collapsed its history."
show_history(assistant.events)

Tag,Event,Content
1..11,Summary,"Original preference: space movie, firm no horror. Compared Interstellar (169 min), Gravity (91 min), Alien (117 min). Recommended Interstellar for expansive non-horror space drama if length acceptable; serious/emotionally intense and over two hours, so not suited to new light comedy / strictly under 120 min request. Gravity retained as shorter non-horror alternative but tense survival thriller with sustained threat/disturbing images, not light time-travel comedy. Alien rejected for horror regardless of runtime. Current shortlist replaces these with Back to the Future, About Time, Edge of Tomorrow; keep no-horror rule.Archived events1 · Task2 · ToolCallEvent3 · PythonOutput4 · LLMResponse5 · ToolCallEvent6 · PythonOutput7 · LLMResponse8 · ToolCallEvent9 · PythonOutput10 · LLMResponse11 · ToolCallEvent"
12,Task,## Task: suggest Help choose a movie. Call lookup_movie() for new candidates and use the catalog's r
13,ToolCallEvent,
14,PythonOutput,"Task: suggest() request (str): ""Actually, let's switch to a light time-travel comedy, strictly under"
15,LLMResponse,
16,ToolCallEvent,
17,PythonOutput,"[('1', 'Task'), ('12', 'Task')] ['1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12', '13'"
18,LLMResponse,
19,ToolCallEvent,
20,PythonOutput,"{ ""title"": ""Back to the Future"", ""year"": 1985, ""runtime_minutes"": 116, ""genres"": [ ""science fiction"""


What happened during that turn? The model generated a call to `self.events.collapse()`. A call might look like this:

```python
self.events.collapse(
    "1", "11",
    "Space shortlist: Interstellar (169 min), Gravity (91 min), Alien (117 min). "
    "No horror: reject Alien. Interstellar is too long and serious; Gravity "
    "is tense rather than funny. Now choose a light time-travel comedy "
    "under two hours, keeping the no-horror rule.",
)
```

The first two arguments mark the event range to archive, including both endpoints. The third is the summary the model writes. The current request stays outside that range, and the originals remain retrievable by tag.

The exact tags and summary vary between runs.

> 💡 Want to see the call the model actually generated? Run `nooa start-dev` in a local terminal before rerunning the conversation, then open the trace viewer at `http://localhost:5001` and inspect the `execute_python` steps.

**“Wait, why did we rule out the first one?”**

Let's ask a follow-up question. With Interstellar's full card archived, can the
agent explain why it no longer fits and confirm its current recommendation?
It can use the summary or retrieve archived events if it needs more detail.
In the trace, check whether it answers without calling `lookup_movie()` again.


In [8]:
final_answer = await assistant.suggest(
    "Wait, why did we rule out the first movie you checked? Remind me of its "
    "runtime and tone, then confirm tonight's winner and its runtime."
)
print(final_answer)

The first movie we checked was **Interstellar (169 minutes)**—an expansive, serious, emotionally intense space drama. It wasn’t ruled out for horror; it simply no longer fits your new brief: it’s over two hours and not a light time-travel comedy.

Tonight’s winner is **Back to the Future (116 minutes)**: playful, energetic, non-horror, and under your strict two-hour limit.

## Recap

- **Expose the Events API:** `spec(self, "events", hidden=False)` lets the model inspect its own history. It is hidden by default.
- **The model manages its context:** it generated a call to `self.events.collapse(start, end, summary)`, which archives a range of events and puts its own summary in their place.
- **Keep decisions, drop details:** the useful memory is our preferences, the movies we considered, and why each option fits or fails. Full plot descriptions can leave the active history.
- **Nothing is lost:** archived events stay retrievable by tag, so the agent can revisit them if a needed fact is missing.

Each run makes fresh model calls, so the wording and collapsed range may vary.